# 🎭 Face Swap Kaggle 教程

**适用平台：** Kaggle Notebooks（免费 GPU：T4 x2 或 P100）

**本教程功能：** 将源人物（source）的脸部替换到目标视频（target）中，生成换脸视频。

## ⚠️ Kaggle 初始设置（在运行本笔记本之前必须完成）

1. **启用 GPU：** 右上角 `Settings` → `Accelerator` → 选择 `GPU T4 x2` 或 `GPU P100`
2. **开启网络：** 右上角 `Settings` → `Internet` → 选择 `Internet on`
3. **添加数据集：** 右上角 `Add input` → 上传 `cloud_gpu_faceswap_upload.tar.gz`（由 Mac 本地 `scripts/00_make_upload_package.sh` 生成）
4. **导入笔记本：** 在 Kaggle 创建新笔记本，粘贴本 notebook 的代码（或直接 import 本 notebook）

⚠️ **注意：** Kaggle 工作区 `/kaggle/working` 的文件为临时存储，会话结束后消失。务必在结束前下载结果。

## Step 0 — 检查 GPU 环境

⏱ 预计时间：几秒钟

确认 Kaggle 分配到 GPU（应为 T4 x2 或 P100）。

In [ ]:
!nvidia-smi

In [ ]:
import os
ROOT = '/kaggle/working/faceswap_work'
os.makedirs(ROOT, exist_ok=True)
print('工作目录:', ROOT)

## Step 1 — 定位并解压上传的数据包

⏱ 预计时间：1–3 分钟

Kaggle 的上传包位于 `/kaggle/input/` 目录下的数据集文件夹中。
此代码会自动搜索并解压 `cloud_gpu_faceswap_upload.tar.gz`。

⚠️ **前提：** 必须先将 `cloud_gpu_faceswap_upload.tar.gz` 作为 Kaggle Dataset 上传（见上方设置步骤）。

In [ ]:
from pathlib import Path

# 搜索上传的数据包（Kaggle 数据集挂载在 /kaggle/input/ 下）
candidates = list(Path('/kaggle/input').rglob('cloud_gpu_faceswap_upload.tar.gz'))
if not candidates:
    raise FileNotFoundError(
        '未找到 cloud_gpu_faceswap_upload.tar.gz\n'
        '请确认已添加 Kaggle Dataset：右上角 Add input → 上传 cloud_gpu_faceswap_upload.tar.gz'
    )
BUNDLE = str(candidates[0])
print('数据包路径:', BUNDLE)

# 解压到 Kaggle 工作区
!rm -rf /kaggle/working/faceswap_work/cloud_gpu_faceswap /kaggle/working/faceswap_work/source_faces
!tar -xzf "$BUNDLE" -C /kaggle/working/faceswap_work

# 显示解压后的文件结构（确认完整性）
!find /kaggle/working/faceswap_work -maxdepth 3 -type f | sort | sed -n '1,80p'

## Step 2 — 安装 Faceswap 工具

⏱ 预计时间：10–20 分钟

从 GitHub 克隆 deepfakes/faceswap 仓库，并安装 NVIDIA CUDA 依赖。

⚠️ **必须开启 Internet：** 在 Kaggle Notebook 设置中启用网络访问，否则无法安装 Python 包。

**包括以下步骤：**
- 安装 `git`（Kaggle 镜像已有 Python）
- 克隆 faceswap 仓库
- 创建 Python 虚拟环境
- 安装 `requirements_nvidia.txt`

⚠️ 如果遇到网络错误，重启笔记本（`Restart & clear cell output`）后重试。

In [ ]:
%%bash
set -euo pipefail
cd /kaggle/working/faceswap_work/cloud_gpu_faceswap

# 克隆 faceswap（已有则跳过）
mkdir -p tools
if [ ! -d tools/faceswap/.git ]; then
  echo '正在克隆 faceswap 仓库 ...'
  git clone https://github.com/deepfakes/faceswap.git tools/faceswap
else
  echo 'faceswap 仓库已存在，跳过克隆'
fi

# 创建 Python 虚拟环境（降级方案：直接用 pip）
cd tools/faceswap
if [ ! -d .venv ]; then
  python3 -m venv .venv 2>/dev/null || python3 -m pip install --user virtualenv && python3 -m virtualenv .venv
fi
source .venv/bin/activate

# 升级 pip 并安装依赖
python -m pip install --upgrade pip setuptools wheel
python -m pip install -r requirements/requirements_nvidia.txt

# 验证安装成功
python faceswap.py -h >/dev/null
echo '\n✅ faceswap 安装完成！'

## Step 3 — 准备素材并提取人脸

⏱ 预计时间：5–15 分钟

此步骤执行：
1. 复制源人物照片到工作区
2. 将目标视频拆解为帧
3. 使用 s3fd 模型检测并裁剪所有人脸

⚠️ **建议在训练前人工检查提取结果**（见下一步），删除错误人脸图。

In [ ]:
%%bash
set -euo pipefail
cd /kaggle/working/faceswap_work/cloud_gpu_faceswap

# 准备素材（复制源图片 + 拆解目标视频）
bash scripts/02_prepare_workspace.sh

# 提取源和目标视频中的人脸
bash scripts/03_extract_faces.sh

## Step 4 — 人工检查提取的人脸

查看下方缩略图拼图，确认：
- **Source Faces**：全部是源人物的脸
- **Target Faces**：全部是目标视频中需要被替换的脸

如果发现错误人脸：
1. 找到 `/kaggle/working/faceswap_work/cloud_gpu_faceswap/workspace/`
2. 进入 `source_faces_extract/` 或 `target_faces_extract/`
3. 删除错误的人脸图

✅ 确认无误后，进入下一步训练。

In [ ]:
from pathlib import Path
from PIL import Image, ImageOps, ImageDraw
from IPython.display import display

def contact_sheet(folder, title, thumb=128, cols=8):
    """生成缩略图拼图，方便快速检查人脸提取结果"""
    paths = sorted(Path(folder).glob('*'))[:64]
    if not paths:
        print('无图片:', folder)
        return
    rows = (len(paths) + cols - 1) // cols
    sheet = Image.new('RGB', (cols * thumb, rows * (thumb + 22)), 'white')
    draw = ImageDraw.Draw(sheet)
    for idx, path in enumerate(paths):
        img = Image.open(path).convert('RGB')
        img = ImageOps.contain(img, (thumb, thumb))
        x = (idx % cols) * thumb
        y = (idx // cols) * (thumb + 22)
        sheet.paste(img, (x, y))
        draw.text((x + 2, y + thumb + 2), path.name[:18], fill=(0, 0, 0))
    print(title)
    display(sheet)

base = '/kaggle/working/faceswap_work/cloud_gpu_faceswap/workspace'
contact_sheet(f'{base}/source_faces_extract', '【源人物】人脸提取结果（Source Faces）')
contact_sheet(f'{base}/target_faces_extract', '【目标人物】人脸提取结果（Target Faces）')

## Step 5 — 训练模型

⏱ 预计时间：
- **1000 次迭代**：15–30 分钟
- **3000 次迭代**：45–90 分钟（推荐）
- **5000 次迭代**：90–180 分钟（高质量）

Kaggle GPU（T4 x2）显存比 Colab 更大，可使用较高的 batch size。

**调整迭代次数（默认 3000）：**
```bash
ITERATIONS=5000 bash scripts/04_train_preview.sh
```

In [ ]:
%%bash
set -euo pipefail
cd /kaggle/working/faceswap_work/cloud_gpu_faceswap

# 训练模型（默认 3000 次迭代）
ITERATIONS=3000 bash scripts/04_train_preview.sh

## Step 6 — 转换并下载换脸视频

⏱ 预计时间：3–10 分钟

使用训练好的模型将源脸替换到目标视频帧，然后合成为 MP4。

⚠️ **Kaggle 存储为临时文件，务必下载！**
下载方式：
- 方式 1：点击右侧 `Output` 面板中的 `faceswap_test.mp4` 下载
- 方式 2：运行下方代码单元格生成下载链接

In [ ]:
%%bash
set -euo pipefail
cd /kaggle/working/faceswap_work/cloud_gpu_faceswap

# 执行换脸转换
bash scripts/05_convert_test.sh

# 复制到 Kaggle 标准输出目录（可在 Output 面板直接下载）
mkdir -p /kaggle/working/faceswap_output
cp -f output/faceswap_test.mp4 /kaggle/working/faceswap_output/faceswap_test.mp4

# 确认输出文件
ls -lh /kaggle/working/faceswap_output/faceswap_test.mp4

In [ ]:
# 生成下载链接（点击链接即可下载到本地）
from IPython.display import FileLink, display
display(FileLink('/kaggle/working/faceswap_output/faceswap_test.mp4'))

---

## ❓ 常见问题（FAQ）

### Q1: Kaggle Notebook 找不到 dataset
**A：** 确认已将 `cloud_gpu_faceswap_upload.tar.gz` 作为 **Dataset**（而非 Code）上传。
在 Notebook 编辑页面：右侧 `Add input` → `Datasets` → 上传或添加 `cloud_gpu_faceswap_upload.tar.gz`。

### Q2: Internet 访问被禁用
**A：** 右上角 `Settings` → `Internet` → 确认为 `Internet on`。新建 Notebook 后首次需要手动开启。

### Q3: GPU 显存不足（OOM）
**A：** 降低 batch size。编辑 `scripts/04_train_preview.sh`，将 `-bs 8` 改为 `-bs 4`。

### Q4: 安装时间过长或超时
**A：** Kaggle Notebook 每次重启环境需要重新安装。如果中断，重启后从 Step 2 继续（已克隆的仓库会被跳过）。

### Q5: 人脸提取结果很差
**A：** 检查源图片质量（建议高清、正面），目标视频中的人脸被严重遮挡会影响提取效果。

### Q6: 如何处理完整视频？
**A：** 在测试视频效果满意后，运行：
```bash
bash scripts/06_convert_full_if_approved.sh
```